# Full model

In [1]:
import unittest
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, TimeDistributed, Bidirectional, GRU, LayerNormalization, Masking
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    LayerNormalization,
    TimeDistributed,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import time
import deepof
import deepof.data

NameError: name 'Callable' is not defined

In [ ]:
4.8793/0.24771088

19.697560317092247

In [ ]:
import math
from typing import Tuple, Callable, Dict

import torch
import torch.nn.functional as F


def _l2_normalize(x: torch.Tensor, dim: int = 1, eps: float = 1e-12) -> torch.Tensor:
    return F.normalize(x, p=2, dim=dim, eps=eps)


def _cosine_similarity_pt(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # x: (N, D), y: (N, D) -> (N, N)
    x1 = x.unsqueeze(1)  # (N, 1, D)
    y1 = y.unsqueeze(0)  # (1, N, D)
    return F.cosine_similarity(x1, y1, dim=2)


def _dot_similarity_pt(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return x @ y.t()


def _euclidean_similarity_pt(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x1 = x.unsqueeze(1)  # (N, 1, D)
    y1 = y.unsqueeze(0)  # (1, N, D)
    d = torch.sqrt(torch.clamp(((x1 - y1) ** 2).sum(dim=2), min=0.0))
    s = 1.0 / (1.0 + d)
    return s


def _edit_similarity_pt(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # Matches provided TF code (same as euclidean similarity transform)
    return _euclidean_similarity_pt(x, y)


_SIMILARITIES: Dict[str, Callable[[torch.Tensor, torch.Tensor], torch.Tensor]] = {
    "cosine": _cosine_similarity_pt,
    "dot": _dot_similarity_pt,
    "euclidean": _euclidean_similarity_pt,
    "edit": _edit_similarity_pt,
}


def _off_diagonal_rows(sim: torch.Tensor) -> torch.Tensor:
    """
    Extract off-diagonal elements row-wise and reshape to (N, N-1),
    mirroring TF's boolean_mask+reshape.
    """
    N = sim.shape[0]
    mask = torch.ones((N, N), dtype=torch.bool, device=sim.device)
    mask.fill_diagonal_(False)
    flat = sim.reshape(-1)
    masked = flat[mask.reshape(-1)]
    return masked.reshape(N, N - 1)


def nce_loss_pt(
    history: torch.Tensor,
    future: torch.Tensor,
    similarity: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    temperature: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Exact port of provided TF nce_loss (including BCE-with-logits on a positive ratio).
    """
    N = history.shape[0]
    sim = similarity(history, future)  # (N, N)
    pos_sim = torch.exp(torch.diag(sim) / temperature)  # (N,)

    neg = _off_diagonal_rows(sim)  # (N, N-1)
    all_sim = torch.exp(sim / temperature)  # (N, N)

    logits = pos_sim.sum() / all_sim.sum(dim=1)  # (N,)
    labels = torch.ones_like(logits)

    bce = torch.nn.BCEWithLogitsLoss(reduction="mean")
    loss = bce(logits, labels)

    mean_sim = torch.diag(sim).mean()
    mean_neg = neg.mean()
    return loss, mean_sim, mean_neg


def dcl_loss_pt(
    history: torch.Tensor,
    future: torch.Tensor,
    similarity: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    temperature: float = 0.1,
    debiased: bool = True,
    tau_plus: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    N = history.shape[0]
    sim = similarity(history, future)  # (N, N)
    pos_sim = torch.exp(torch.diag(sim) / temperature)  # (N,)

    neg = _off_diagonal_rows(sim)  # (N, N-1)
    neg_sim = torch.exp(neg / temperature)  # (N, N-1)

    if debiased:
        N_eff = N - 1
        Ng = (-tau_plus * N_eff * pos_sim + neg_sim.sum(dim=-1)) / (1.0 - tau_plus)
        min_clip = N_eff * math.e ** (-1.0 / temperature)
        Ng = torch.clamp(Ng, min=min_clip, max=torch.finfo(history.dtype).max)
    else:
        Ng = neg_sim.sum(dim=-1)

    loss = (-torch.log(pos_sim / (pos_sim + Ng))).mean()

    mean_sim = torch.diag(sim).mean()
    mean_neg = neg.mean()
    return loss, mean_sim, mean_neg


def fc_loss_pt(
    history: torch.Tensor,
    future: torch.Tensor,
    similarity: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    temperature: float = 0.1,
    elimination_topk: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    N = history.shape[0]
    elim = min(elimination_topk, 0.5)
    k = int(math.ceil(elim * N))

    sim = similarity(history, future) / temperature  # (N, N)
    pos_sim = torch.exp(torch.diag(sim))  # (N,)

    neg_sim_raw = _off_diagonal_rows(sim)  # (N, N-1)
    sorted_sim, _ = torch.sort(neg_sim_raw, dim=1)  # ascending

    if k == 0:
        k = 1
    keep = (N - 1) - k
    trimmed = sorted_sim[:, : max(keep, 0)]  # may be empty

    neg_sim = torch.exp(trimmed).sum(dim=1) if trimmed.numel() > 0 else torch.zeros(
        N, device=sim.device, dtype=sim.dtype
    )

    loss = (-torch.log(pos_sim / (pos_sim + neg_sim))).mean()

    mean_sim = torch.diag(sim).mean() * temperature
    mean_neg = trimmed.mean() * temperature if trimmed.numel() > 0 else torch.tensor(
        0.0, device=sim.device, dtype=sim.dtype
    )
    return loss, mean_sim, mean_neg


def hard_loss_pt(
    history: torch.Tensor,
    future: torch.Tensor,
    similarity: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    temperature: float,
    beta: float = 0.0,
    debiased: bool = True,
    tau_plus: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    N = history.shape[0]
    sim = similarity(history, future)  # (N, N)

    pos_sim = torch.exp(torch.diag(sim) / temperature)  # (N,)
    neg = _off_diagonal_rows(sim)  # (N, N-1)
    neg_sim = torch.exp(neg / temperature)  # (N, N-1)

    if beta == 0.0:
        reweight = torch.ones_like(neg_sim)
    else:
        denom = neg_sim.mean(dim=1, keepdim=True)
        reweight = (beta * neg_sim) / denom

    if debiased:
        N_eff = N - 1
        Ng = (-tau_plus * N_eff * pos_sim + (reweight * neg_sim).sum(dim=-1)) / (
            1.0 - tau_plus
        )
        min_clip = math.e ** (-1.0 / temperature)
        Ng = torch.clamp(Ng, min=min_clip, max=torch.finfo(history.dtype).max)
    else:
        Ng = neg_sim.sum(dim=-1)

    loss = (-torch.log(pos_sim / (pos_sim + Ng))).mean()

    mean_sim = torch.diag(sim).mean()
    mean_neg = neg.mean()
    return loss, mean_sim, mean_neg


def select_contrastive_loss_pt(
    history: torch.Tensor,
    future: torch.Tensor,
    similarity: str,
    loss_fn: str = "nce",
    temperature: float = 0.1,
    tau: float = 0.1,
    beta: float = 0.1,
    elimination_topk: float = 0.1,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    sim_fn = _SIMILARITIES[similarity]

    if loss_fn == "nce":
        return nce_loss_pt(history, future, sim_fn, temperature)
    elif loss_fn == "dcl":
        return dcl_loss_pt(history, future, sim_fn, temperature, debiased=True, tau_plus=tau)
    elif loss_fn == "fc":
        return fc_loss_pt(history, future, sim_fn, temperature, elimination_topk=elimination_topk)
    elif loss_fn == "hard_dcl":
        return hard_loss_pt(history, future, sim_fn, temperature, beta=beta, debiased=True, tau_plus=tau)
    else:
        raise ValueError(f"Unknown loss_fn: {loss_fn}")

In [ ]:
# deepof/clustering/models_new.py (additions)

from typing import Tuple, Optional, Dict, Any
import torch
import torch.nn as nn

#from deepof.clustering.model_utils_new import (
#    _l2_normalize,
#    select_contrastive_loss_pt,
#    _SIMILARITIES,  # for debug
#)


class ContrastivePT(nn.Module):
    """
    PyTorch port of the TF Contrastive model.

    Inputs:
      x: (B, T, N, F)
      a: (B, T, E, F_edge)

    Behavior:
      - Builds an encoder for sequences of length T//2.
      - forward(x_half, a_half) returns embeddings (B, D) for a given half-window.
      - compute_loss(x_full, a_full) slices pos/neg windows and returns (loss, pos_mean, neg_mean, debug).
    """

    def __init__(
        self,
        input_shape: Tuple[int, int, int],         # (T, N, F)
        edge_feature_shape: Tuple[int, int, int],  # (T, E, F_edge)
        adjacency_matrix,
        encoder_type: str = "TCN",
        latent_dim: int = 8,
        use_gnn: bool = True,
        temperature: float = 0.1,
        similarity_function: str = "cosine",
        loss_function: str = "nce",
        beta: float = 0.1,
        tau: float = 0.1,
        interaction_regularization: float = 0.0,
    ):
        super().__init__()

        T, N, F_in = input_shape
        Te, E, Fe = edge_feature_shape
        if T != Te:
            raise ValueError(f"Node and edge time dims must match: T={T}, Te={Te}")
        if T < 2 or (T % 2) != 0:
            raise ValueError(
                f"ContrastivePT requires an even sequence length T>=2. Got T={T}. "
                "Please pre-trim or pad your sequences (e.g., use T=24 if original T=25)."
            )

        self.full_time_steps = T
        self.window_length = T // 2
        self.input_shape = input_shape
        self.edge_feature_shape = edge_feature_shape
        self.adjacency_matrix = adjacency_matrix

        self.latent_dim = latent_dim
        self.use_gnn = use_gnn
        self.encoder_type = encoder_type

        self.temperature = temperature
        self.similarity_function = similarity_function
        self.loss_function = loss_function
        self.beta = beta
        self.tau = tau
        self.interaction_regularization = interaction_regularization

        if encoder_type == "recurrent":
            self.encoder = deepof.clustering.models_new.RecurrentEncoderPT(
                input_shape=(self.window_length, N, F_in),
                edge_feature_shape=(self.window_length, E, Fe),
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
        elif encoder_type == "TCN":
            self.encoder = deepof.clustering.models_new.TCNEncoderPT(
                input_shape=(self.window_length, N, F_in),
                edge_feature_shape=(self.window_length, E, Fe),
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
        elif encoder_type == "transformer":
            self.encoder = deepof.clustering.models_new.TFMEncoderPT(
                input_shape=(self.window_length, N, F_in),
                edge_feature_shape=(self.window_length, E, Fe),
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
            )
        else:
            raise ValueError(f"Unknown encoder_type: {encoder_type}")

        # Debug cache
        self._last_debug: Dict[str, Any] = {}

    def forward(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """
        Encode a half-window:
          x: (B, T_half, N, F), a: (B, T_half, E, Fe) -> (B, D)
        """
        return self.encoder(x, a)

    @staticmethod
    def _ts_samples(x: torch.Tensor, win: int):
        # TF parity: pos = x[:, 1:win+1], neg = x[:, -win:]
        pos = x[:, 1 : win + 1]
        neg = x[:, -win:]
        return pos, neg

    def compute_loss(
        self,
        x: torch.Tensor,  # (B, T, N, F)
        a: torch.Tensor,  # (B, T, E, Fe)
        return_debug: bool = False,
    ):
        B, T, N, F_in = x.shape
        if T != self.full_time_steps:
            raise ValueError(f"Input time dim T={T} does not match model T={self.full_time_steps}")

        # Slice windows exactly like TF
        pos_x, neg_x = self._ts_samples(x, self.window_length)
        pos_a, neg_a = self._ts_samples(a, self.window_length)

        # Encode and normalize
        z_pos = self.encoder(pos_x, pos_a)  # (B, D)
        z_neg = self.encoder(neg_x, neg_a)  # (B, D)
        z_pos = _l2_normalize(z_pos, dim=1, eps=1e-12)
        z_neg = _l2_normalize(z_neg, dim=1, eps=1e-12)

        # Compute loss
        loss, pos_mean, neg_mean = select_contrastive_loss_pt(
            z_pos,
            z_neg,
            similarity=self.similarity_function,
            loss_fn=self.loss_function,
            temperature=self.temperature,
            tau=self.tau,
            beta=self.beta,
            elimination_topk=0.1,  # same default as TF snippet
        )

        debug = None
        if return_debug:
            # Build a minimal debug pack for parity troubleshooting
            sim_fn = _SIMILARITIES[self.similarity_function]
            with torch.no_grad():
                sim = sim_fn(z_pos, z_neg)  # (B, B)
                diag = torch.diag(sim)
                offdiag = sim[~torch.eye(B, dtype=torch.bool, device=sim.device)]
                offdiag = offdiag.view(B, B - 1) if B > 1 else offdiag.view(B, 0)

                debug = {
                    "z_pos_shape": torch.tensor(z_pos.shape),
                    "z_neg_shape": torch.tensor(z_neg.shape),
                    "z_pos_norm_mean": torch.norm(z_pos, dim=1).mean().cpu(),
                    "z_neg_norm_mean": torch.norm(z_neg, dim=1).mean().cpu(),
                    "sim_diag_mean": diag.mean().cpu(),
                    "sim_offdiag_mean": offdiag.mean().cpu() if offdiag.numel() > 0 else torch.tensor(0.0),
                    "loss": loss.detach().cpu(),
                    "pos_mean": pos_mean.detach().cpu(),
                    "neg_mean": neg_mean.detach().cpu(),
                }
            self._last_debug = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in debug.items()}

        return loss, pos_mean, neg_mean, debug

    def get_last_debug(self) -> Dict[str, Any]:
        return self._last_debug


def training_step_contrastive_pt(
    model: ContrastivePT,
    x: torch.Tensor,
    a: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    clip_value: float = 0.75,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss, pos_mean, neg_mean, _ = model.compute_loss(x, a, return_debug=False)
    loss.backward()
    torch.nn.utils.clip_grad_value_(model.parameters(), clip_value)
    optimizer.step()
    return loss.item(), float(pos_mean.item()), float(neg_mean.item())

In [ ]:
def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch):
    # Find CensNetConv and its inbound tensors
    gnn_tf = next(l for l in tf_enc.layers if l.__class__.__name__ == "CensNetConv")
    node = gnn_tf._inbound_nodes[-1]  # use last call
    in_ts = getattr(node, "input_tensors", None)
    if in_ts is None:
        in_ts = getattr(node, "inputs")
    in_ts = _flatten_keras_tensors(in_ts)

    # Among inbound tensors, pick the two with rank 3 and last dim == 2*latent
    cand = [t for t in in_ts if len(t.shape) == 3]
    # Disambiguate by producer: nested Functional input feature dim
    pairs = []
    for t in cand:
        prod = _layer_from_tensor(t)   # nested Functional (recurrent block)
        if isinstance(prod, tf.keras.Model):
            in_last = int(prod.inputs[0].shape[-1])
            pairs.append((t, prod, in_last))

    # Map by input feature dim
    tf_node_pre_t = next(t for t, prod, in_last in pairs if in_last == node_in_ch)
    tf_edge_pre_t = next(t for t, prod, in_last in pairs if in_last == edge_in_ch)
    tf_node_block = _layer_from_tensor(tf_node_pre_t)
    tf_edge_block = _layer_from_tensor(tf_edge_pre_t)
    return gnn_tf, tf_node_pre_t, tf_edge_pre_t, tf_node_block, tf_edge_block

def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)


import numpy as np
import torch
import tensorflow as tf
from tensorflow.keras.layers import TimeDistributed, Conv1D, Bidirectional, LayerNormalization, Dense


def _to_torch(x):
    return torch.from_numpy(np.asarray(x))


def _extract_conv1d_weight(conv_layer):
    # Keras Conv1D kernel: (k, in_c, out_c)
    w = conv_layer.get_weights()[0]
    if w.ndim == 3:
        # -> PyTorch Conv1d: (out_c, in_c, k)
        return np.transpose(w, (2, 1, 0))
    elif w.ndim == 4 and w.shape[1] == 1:
        # Sometimes shows as (k, 1, in_c, out_c)
        w = np.squeeze(w, axis=1)
        return np.transpose(w, (2, 1, 0))
    else:
        raise RuntimeError(f"Unexpected Conv1D kernel shape: {w.shape}")


def _extract_gru_raw_weights(gru_layer):
    """
    Return (W_ih, W_hh, B_ih, B_hh) from a Keras GRU layer, normalized to:
      W_ih: (in, 3*units), W_hh: (units, 3*units)
      B_ih: (3*units,), B_hh: (3*units,)
    Handles reset_after=True case where bias can be 2D (2, 3*units).
    """
    ws = gru_layer.get_weights()
    units = gru_layer.units

    if len(ws) == 3:
        W_ih, W_hh, B = ws
        # B can be:
        # - flat (2*3*units,)
        # - 2D (2, 3*units)
        # - flat (3*units,) if reset_after=False (rare)
        if B.ndim == 2 and B.shape[0] == 2 and B.shape[1] == 3 * units:
            B_ih, B_hh = B[0].copy(), B[1].copy()
        elif B.ndim == 1 and B.shape[0] == 2 * 3 * units:
            half = 3 * units
            B_ih, B_hh = B[:half].copy(), B[half:].copy()
        elif B.ndim == 1 and B.shape[0] == 3 * units:
            # No recurrent bias; provide zeros for recurrent bias to match PyTorch expectations
            B_ih, B_hh = B.copy(), np.zeros_like(B)
        else:
            raise RuntimeError(f"Unexpected GRU bias shape: {B.shape}")
    elif len(ws) == 4:
        W_ih, W_hh, B_ih, B_hh = ws
        # Ensure 1D
        B_ih = B_ih.reshape(-1).copy()
        B_hh = B_hh.reshape(-1).copy()
    else:
        raise RuntimeError(f"Unexpected number of GRU weights: {len(ws)}")

    return W_ih, W_hh, B_ih, B_hh


def _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units):
    """
    Keras gate order: z, r, n -> PyTorch: r, z, n.
    Use explicit slicing by units (no np.split to avoid shape issues).
    Returns:
      W_ih_pt: (3h, in), W_hh_pt: (3h, h), B_ih_pt: (3h,), B_hh_pt: (3h,)
    """
    # W_ih: (in, 3*units)
    W_ih_z = W_ih[:, 0 * units:1 * units]
    W_ih_r = W_ih[:, 1 * units:2 * units]
    W_ih_n = W_ih[:, 2 * units:3 * units]
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1).T  # -> (3*units, in)

    # W_hh: (units, 3*units)
    W_hh_z = W_hh[:, 0 * units:1 * units]
    W_hh_r = W_hh[:, 1 * units:2 * units]
    W_hh_n = W_hh[:, 2 * units:3 * units]
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1).T  # -> (3*units, units)

    # Biases: (3*units,)
    B_ih_z = B_ih[0 * units:1 * units]
    B_ih_r = B_ih[1 * units:2 * units]
    B_ih_n = B_ih[2 * units:3 * units]
    B_hh_z = B_hh[0 * units:1 * units]
    B_hh_r = B_hh[1 * units:2 * units]
    B_hh_n = B_hh[2 * units:3 * units]

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])  # (3*units,)
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])  # (3*units,)

    return W_ih_pt, W_hh_pt, B_ih_pt, B_hh_pt


def transfer_recurrent_block_weights(tf_block_model, pt_block):
    """
    Minimal transfer for a single recurrent block (node or edge):
      - TimeDistributed(Conv1D)
      - TimeDistributed(Bidirectional(GRU)) x 2
      - LayerNormalization x 2
    """
    # Conv1D (inside TimeDistributed)
    td_convs = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Conv1D)]
    assert len(td_convs) >= 1, "No TimeDistributed(Conv1D) found in recurrent block"
    conv_td = td_convs[0]
    conv_w_pt = _extract_conv1d_weight(conv_td.layer)
    pt_block.conv1d.weight.data = _to_torch(conv_w_pt)
    if len(conv_td.layer.get_weights()) > 1 and pt_block.conv1d.bias is not None:
        pt_block.conv1d.bias.data = _to_torch(conv_td.layer.get_weights()[1])

    # Two TimeDistributed(Bidirectional(GRU))
    td_bis = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Bidirectional)]
    assert len(td_bis) >= 2, f"Expected 2 TimeDistributed(Bidirectional) GRUs, got {len(td_bis)}"
    bi1, bi2 = td_bis[0].layer, td_bis[1].layer

    # Norm layers
    norms = [l for l in tf_block_model.layers if isinstance(l, LayerNormalization)]
    assert len(norms) >= 2, f"Expected 2 LayerNormalization layers, got {len(norms)}"

    # First BiGRU -> pt_block.gru1 + norm1
    units1 = bi1.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.forward_layer)
    W_ih_f, W_hh_f, B_ih_f, B_hh_f = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0.data = _to_torch(W_ih_f)
    pt_block.gru1.weight_hh_l0.data = _to_torch(W_hh_f)
    pt_block.gru1.bias_ih_l0.data = _to_torch(B_ih_f)
    pt_block.gru1.bias_hh_l0.data = _to_torch(B_hh_f)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.backward_layer)
    W_ih_b, W_hh_b, B_ih_b, B_hh_b = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0_reverse.data = _to_torch(W_ih_b)
    pt_block.gru1.weight_hh_l0_reverse.data = _to_torch(W_hh_b)
    pt_block.gru1.bias_ih_l0_reverse.data = _to_torch(B_ih_b)
    pt_block.gru1.bias_hh_l0_reverse.data = _to_torch(B_hh_b)

    pt_block.norm1.weight.data = _to_torch(norms[0].get_weights()[0])
    pt_block.norm1.bias.data = _to_torch(norms[0].get_weights()[1])

    # Second BiGRU -> pt_block.gru2 + norm2
    units2 = bi2.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.forward_layer)
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0.data = _to_torch(W_ih_f2)
    pt_block.gru2.weight_hh_l0.data = _to_torch(W_hh_f2)
    pt_block.gru2.bias_ih_l0.data = _to_torch(B_ih_f2)
    pt_block.gru2.bias_hh_l0.data = _to_torch(B_hh_f2)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.backward_layer)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0_reverse.data = _to_torch(W_ih_b2)
    pt_block.gru2.weight_hh_l0_reverse.data = _to_torch(W_hh_b2)
    pt_block.gru2.bias_ih_l0_reverse.data = _to_torch(B_ih_b2)
    pt_block.gru2.bias_hh_l0_reverse.data = _to_torch(B_hh_b2)

    pt_block.norm2.weight.data = _to_torch(norms[1].get_weights()[0])
    pt_block.norm2.bias.data = _to_torch(norms[1].get_weights()[1])


def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Spektral CensNetConv -> CensNetConvPT:
      Expected TF order: node_kernel, edge_kernel, node_weights, edge_weights, node_bias, edge_bias
    """
    kn_tf, ke_tf, pn_tf, pe_tf, bn_tf, be_tf = tf_layer.get_weights()
    pt_layer.node_kernel.data = _to_torch(kn_tf)
    pt_layer.edge_kernel.data = _to_torch(ke_tf)

    pn_tf = pn_tf.reshape(-1, 1) if pn_tf.ndim == 1 else pn_tf
    pe_tf = pe_tf.reshape(-1, 1) if pe_tf.ndim == 1 else pe_tf
    pt_layer.node_weights.data = _to_torch(pn_tf)
    pt_layer.edge_weights.data = _to_torch(pe_tf)

    pt_layer.node_bias.data = _to_torch(bn_tf)
    pt_layer.edge_bias.data = _to_torch(be_tf)


def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def transfer_recurrent_encoder_weights(tf_model, pt_model, verbose=False):
    try:
        tf_enc = tf_model.get_layer("recurrent_encoder")
    except Exception:
        tf_enc = tf_model

    # Final Dense
    w, b = tf_enc.layers[-1].get_weights()
    pt_model.final_dense.weight.data = torch.from_numpy(w.T)
    pt_model.final_dense.bias.data = torch.from_numpy(b)

    if not getattr(pt_model, "use_gnn", False):
        sub = [l for l in tf_enc.layers if isinstance(l, tf.keras.Model)][0]
        transfer_recurrent_block_weights(sub, pt_model.recurrent_block)
        return

    node_in_ch = int(pt_model.node_recurrent_block.conv1d.in_channels)
    edge_in_ch = int(pt_model.edge_recurrent_block.conv1d.in_channels)
    gnn_tf, _, _, tf_node_block, tf_edge_block = _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch)

    transfer_recurrent_block_weights(tf_node_block, pt_model.node_recurrent_block)
    transfer_recurrent_block_weights(tf_edge_block, pt_model.edge_recurrent_block)
    transfer_censnet_weights(gnn_tf, pt_model.spatial_gnn_block)

    if verbose:
        print(f"Transferred node from {tf_node_block.name} (in_ch={node_in_ch}), edge from {tf_edge_block.name} (in_ch={edge_in_ch})")


In [ ]:
# tests/test_contrastive_translation.py

import unittest
import numpy as np
import torch
import tensorflow as tf

#from deepof.clustering.models_new import ContrastivePT
#from deepof.clustering.model_utils_new import (
#    _l2_normalize,
#    _cosine_similarity_pt,
#    _dot_similarity_pt,
#    _euclidean_similarity_pt,
#    _edit_similarity_pt,
#    select_contrastive_loss_pt,
#)

# Reuse your existing transfer function
# from deepof.clustering.weight_transfer import transfer_recurrent_encoder_weights
#from transfer_utils import transfer_recurrent_encoder_weights  # <-- update path


class TestContrastiveTranslation(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()
        np.random.seed(0)
        torch.manual_seed(0)

        # Use even T to satisfy ContrastivePT assert
        self.B = 8
        self.T = 24
        self.N = 11
        self.E = 11
        self.Fn = 3
        self.Fe = 1
        self.latent_dim = 6

        self.input_shape_tf = (self.B, self.T, self.N * self.Fn)
        self.edge_shape_tf = (self.B, self.T, self.E * self.Fe)

        self.input_shape_pt = (self.T, self.N, self.Fn)
        self.edge_shape_pt = (self.T, self.E, self.Fe)

        # Random symmetric adjacency
        m = np.zeros((self.N, self.N), dtype=np.float32)
        ui = np.triu_indices(self.N)
        c = np.random.choice(len(ui[0]), self.E, replace=True)
        m[ui[0][c], ui[1][c]] = 1.0
        self.A = (m + m.T).astype(np.float32)
        self.A = np.clip(self.A,0,1) #clip double counts on diagonale

        # Random data
        self.x_tf = np.random.randn(*self.input_shape_tf).astype(np.float32)
        self.a_tf = np.random.randn(*self.edge_shape_tf).astype(np.float32)
        self.x_pt = self.x_tf.reshape(self.B, self.T, self.N, self.Fn)
        self.a_pt = self.a_tf.reshape(self.B, self.T, self.E, self.Fe)

        # TF model
        from deepof.models import Contrastive as TFContrastive
        self.tf_model = TFContrastive(
            input_shape=self.input_shape_tf,
            edge_feature_shape=self.edge_shape_tf,
            adjacency_matrix=self.A,
            encoder_type="recurrent",
            latent_dim=self.latent_dim,
            use_gnn=True,
            temperature=0.1,
            similarity_function="cosine",
            loss_function="nce",
            beta=0.1,
            tau=0.1,
            interaction_regularization=0.0,
        )

        # PT model
        self.pt_model = ContrastivePT(
            input_shape=self.input_shape_pt,
            edge_feature_shape=self.edge_shape_pt,
            adjacency_matrix=self.A,
            encoder_type="recurrent",
            latent_dim=self.latent_dim,
            use_gnn=True,
            temperature=0.1,
            similarity_function="cosine",
            loss_function="nce",
            beta=0.1,
            tau=0.1,
        )
        self.pt_model.eval()

        # Windows for encoder parity
        win = self.T // 2
        self.pos_tf = tf.constant(self.x_tf[:, 1:win + 1])
        self.pos_a_tf = tf.constant(self.a_tf[:, 1:win + 1])
        self.pos_pt = torch.from_numpy(self.x_pt[:, 1:win + 1])
        self.pos_a_pt = torch.from_numpy(self.a_pt[:, 1:win + 1])

    def test_01_encoder_exact_match_after_transfer(self):
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder, verbose=False)

        tf_enc = self.tf_model.encoder([self.pos_tf, self.pos_a_tf]).numpy()
        with torch.no_grad():
            pt_enc = self.pt_model.encoder(self.pos_pt, self.pos_a_pt).cpu().numpy()

        np.testing.assert_allclose(tf_enc, pt_enc, rtol=1e-4, atol=1e-5)

    def test_02_similarity_parity(self):
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder, verbose=False)

        tf_enc = self.tf_model.encoder([self.pos_tf, self.pos_a_tf]).numpy()
        pt_enc = self.pt_model.encoder(self.pos_pt, self.pos_a_pt).detach().cpu().numpy()

        tf_enc = tf_enc / (np.linalg.norm(tf_enc, axis=1, keepdims=True) + 1e-12)
        pt_enc = pt_enc / (np.linalg.norm(pt_enc, axis=1, keepdims=True) + 1e-12)

        from deepof.model_utils import (
            _cosine_similarity as tf_cos,
            _dot_similarity as tf_dot,
            _euclidean_similarity as tf_euc,
            _edit_similarity as tf_edt,
        )
        tf_cos_m = tf_cos(tf_enc, tf_enc).numpy()
        tf_dot_m = tf_dot(tf_enc, tf_enc).numpy()
        tf_euc_m = tf_euc(tf_enc, tf_enc).numpy()
        tf_edt_m = tf_edt(tf_enc, tf_enc).numpy()

        pt_cos_m = _cosine_similarity_pt(torch.from_numpy(pt_enc), torch.from_numpy(pt_enc)).numpy()
        pt_dot_m = _dot_similarity_pt(torch.from_numpy(pt_enc), torch.from_numpy(pt_enc)).numpy()
        pt_euc_m = _euclidean_similarity_pt(torch.from_numpy(pt_enc), torch.from_numpy(pt_enc)).numpy()
        pt_edt_m = _edit_similarity_pt(torch.from_numpy(pt_enc), torch.from_numpy(pt_enc)).numpy()

        np.testing.assert_allclose(tf_cos_m, pt_cos_m, rtol=1e-5, atol=1e-6)
        np.testing.assert_allclose(tf_dot_m, pt_dot_m, rtol=1e-5, atol=1e-6)
        np.testing.assert_allclose(tf_euc_m, pt_euc_m, rtol=1e-5, atol=1e-6)
        np.testing.assert_allclose(tf_edt_m, pt_edt_m, rtol=1e-5, atol=1e-6)

    def test_03_losses_parity_all_modes(self):
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder, verbose=False)

        win = self.T // 2
        pos_tf = self.x_tf[:, 1:win + 1]
        neg_tf = self.x_tf[:, -win:]
        pos_a_tf = self.a_tf[:, 1:win + 1]
        neg_a_tf = self.a_tf[:, -win:]

        zpos_tf = self.tf_model.encoder([pos_tf, pos_a_tf]).numpy()
        zneg_tf = self.tf_model.encoder([neg_tf, neg_a_tf]).numpy()
        zpos_tf = zpos_tf / (np.linalg.norm(zpos_tf, axis=1, keepdims=True) + 1e-12)
        zneg_tf = zneg_tf / (np.linalg.norm(zneg_tf, axis=1, keepdims=True) + 1e-12)

        pos_pt = torch.from_numpy(self.x_pt[:, 1:win + 1])
        neg_pt = torch.from_numpy(self.x_pt[:, -win:])
        pos_a_pt = torch.from_numpy(self.a_pt[:, 1:win + 1])
        neg_a_pt = torch.from_numpy(self.a_pt[:, -win:])

        with torch.no_grad():
            zpos_pt = self.pt_model.encoder(pos_pt, pos_a_pt)
            zneg_pt = self.pt_model.encoder(neg_pt, neg_a_pt)
            zpos_pt = _l2_normalize(zpos_pt, dim=1, eps=1e-12)
            zneg_pt = _l2_normalize(zneg_pt, dim=1, eps=1e-12)

        from deepof.model_utils import select_contrastive_loss as select_tf

        for loss_fn in ["nce", "dcl", "fc", "hard_dcl"]:
            tf_loss, tf_pos_m, tf_neg_m = select_tf(
                zpos_tf, zneg_tf,
                similarity="cosine",
                loss_fn=loss_fn,
                temperature=0.1,
                tau=0.1,
                beta=0.1,
                elimination_topk=0.1,
            )
            pt_loss, pt_pos_m, pt_neg_m = select_contrastive_loss_pt(
                zpos_pt, zneg_pt,
                similarity="cosine",
                loss_fn=loss_fn,
                temperature=0.1,
                tau=0.1,
                beta=0.1,
                elimination_topk=0.1,
            )

            np.testing.assert_allclose(tf_loss.numpy(), pt_loss.cpu().numpy(), rtol=1e-5, atol=1e-6)
            np.testing.assert_allclose(tf_pos_m.numpy(), pt_pos_m.cpu().numpy(), rtol=1e-5, atol=1e-6)
            np.testing.assert_allclose(tf_neg_m.numpy(), pt_neg_m.cpu().numpy(), rtol=1e-5, atol=1e-6)

    def test_04_training_step_with_nadam_and_clipping(self):
        # Simple smoke test for optimizer parity
        opt = torch.optim.NAdam(self.pt_model.parameters(), lr=1e-3)
        x = torch.from_numpy(self.x_pt)
        a = torch.from_numpy(self.a_pt)
        self.pt_model.train()
        loss0, pos0, neg0 = None, None, None
        for _ in range(2):
            opt.zero_grad(set_to_none=True)
            loss, pos_m, neg_m, _ = self.pt_model.compute_loss(x, a, return_debug=False)
            loss.backward()
            torch.nn.utils.clip_grad_value_(self.pt_model.parameters(), 0.75)
            opt.step()
            loss0, pos0, neg0 = float(loss.item()), float(pos_m.item()), float(neg_m.item())
        self.assertTrue(np.isfinite(loss0))

    def test_05_debug_payload(self):
        x = torch.from_numpy(self.x_pt)
        a = torch.from_numpy(self.a_pt)
        _, _, _, dbg = self.pt_model.compute_loss(x, a, return_debug=True)
        self.assertIn("z_pos_shape", dbg)
        self.assertIn("sim_diag_mean", dbg)
        self.assertIn("loss", dbg)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestContrastiveTranslation)
    unittest.TextTestRunner(verbosity=2).run(suite)

test_01_encoder_exact_match_after_transfer (__main__.TestContrastiveTranslation) ... ok
test_02_similarity_parity (__main__.TestContrastiveTranslation) ... ok
test_03_losses_parity_all_modes (__main__.TestContrastiveTranslation) ... ok
test_04_training_step_with_nadam_and_clipping (__main__.TestContrastiveTranslation) ... ok
test_05_debug_payload (__main__.TestContrastiveTranslation) ... ok

----------------------------------------------------------------------
Ran 5 tests in 26.672s

OK
